# 🌊 Streaming — token-by-token output with `streamSimple`

Notebook 01 used `completeSimple`: you `await` a single call and get the whole reply at once. That's simple, but the user stares at a blank screen until the model finishes.

**Streaming** flips that around. `models.streamSimple(model, context)` returns an [`AssistantMessageEventStream`](https://github.com/earendil-works/pi) — an object that is **both async-iterable** (so you can `for await` over incremental events) **and** carries a `.result()` promise for the final assembled message. You render text as it arrives, then still get the same tidy `AssistantMessage` (with `usage`, `stopReason`, …) at the end.

> **Kernel:** Deno (pi.dev is ESM-only TypeScript; the Deno kernel runs it natively with top-level `await`). See `../README.md`.

## 1. Load env & register Azure OpenAI

Same two-step bootstrap every notebook uses: walk up the tree for a `.env` (keys stay **outside** the repo), then register Azure via the shared `registerAzure()` helper — which folds notebook 01's ~40 lines of provider setup into one call.

In [ ]:
import { loadEnvUp } from "playground/env";

// Loads EVERY .env from this folder up to the filesystem root (closest wins),
// printing the keys loaded from each file.
const env = await loadEnvUp();
console.log(`\n${env.files.length} .env file(s) found; ${env.loaded.length} variable(s) loaded in total.`);

In [ ]:
import { registerAzure } from "playground/azure";

// One call replaces the whole Azure-registration ceremony from notebook 01:
// it reads AZURE_PI_TEST_* from the env, registers the provider, and returns
// the Models collection plus the first deployment resolved and ready to use.
const { models, model, modelId } = registerAzure();
console.log(`Ready to stream from azure-openai/${modelId}.`);

## 2. Build a `Context`

The `Context` is identical to notebook 01 — streaming changes *how* you call the model, not *what* you send. We ask for a few sentences so there's enough text to watch stream in.

In [ ]:
import { type Context } from "@earendil-works/pi-ai";

const context: Context = {
  systemPrompt: "You are a friendly assistant. Keep answers concise.",
  messages: [
    {
      role: "user",
      content: "Explain what token streaming is, in 3 short sentences.",
      timestamp: Date.now(),
    },
  ],
};
console.log("Context ready ✔");

## 3. Stream it — the event loop

`streamSimple` emits a sequence of typed events. Iterate them with `for await` and `switch` on `event.type`. The ones you'll see for a plain text reply:

| Event | Meaning |
|-------|---------|
| `start` | Generation began; `event.partial` is the in-progress `AssistantMessage`. |
| `text_start` | A text block opened. |
| `text_delta` | An incremental chunk of text — `event.delta`. **This is the live output.** |
| `text_end` | The text block finished. |
| `done` | Finished successfully; `event.reason` is the stop reason. |
| `error` | Failed/aborted; `event.error` is the final message with `errorMessage`. |

(There are also `thinking_*` and `toolcall_*` events — those show up in later notebooks on reasoning and tools.)

We write each `text_delta` straight to stdout with **no newline**, so the answer materializes token-by-token instead of appearing all at once.

In [ ]:
// `streamSimple` returns an AssistantMessageEventStream: async-iterable here,
// and carrying a `.result()` promise we use in the next cell.
const stream = models.streamSimple(model, context);

const encoder = new TextEncoder();
const write = (s: string) => Deno.stdout.writeSync(encoder.encode(s));

for await (const event of stream) {
  switch (event.type) {
    case "start":
      console.log(`▶️  streaming from ${event.partial.model}…\n`);
      break;
    case "text_start":
      write("🤖 ");
      break;
    case "text_delta":
      // Print each chunk live — this is what makes the output appear to "type".
      write(event.delta);
      break;
    case "text_end":
      write("\n");
      break;
    case "done":
      console.log(`\n✅ done (reason: ${event.reason})`);
      break;
    case "error":
      console.log(`\n❌ error: ${event.error.errorMessage ?? event.reason}`);
      break;
  }
}

## 4. The final message via `.result()`

Streaming doesn't cost you the structured result. After (or even during) iteration, `await stream.result()` resolves to the complete `AssistantMessage` — the exact same shape `completeSimple` returns — with the assembled `content` blocks, `stopReason`, and token/cost `usage`.

In [ ]:
const final = await stream.result();

// Reassemble the text blocks (the stream already printed them live above).
const text = final.content
  .filter((b) => b.type === "text")
  .map((b) => (b as { text: string }).text)
  .join("");

console.log("--- final AssistantMessage ---");
console.log("stopReason:", final.stopReason);
if (final.errorMessage) console.log("errorMessage:", final.errorMessage);
console.log(
  `tokens: ${final.usage.input} in / ${final.usage.output} out` +
    `  |  cost: $${final.usage.cost.total.toFixed(6)}`,
);
console.log(`\nassembled text (${text.length} chars):\n${text}`);

### Streaming vs. `completeSimple`

| | `completeSimple` (nb 01) | `streamSimple` (this nb) |
|---|---|---|
| Returns | `Promise<AssistantMessage>` | async-iterable stream + `.result()` |
| Output timing | all at once, when done | incrementally, as generated |
| Best for | scripts, batch, tool loops | chat UIs, long replies, perceived speed |
| Final message | the awaited value | `await stream.result()` |

Same `model`, same `Context`, same final `AssistantMessage` — you only change the call and add an event loop.

## ✅ What just happened

1. `loadEnvUp()` + `registerAzure()` bootstrapped the Azure OpenAI provider (as in nb 01, now one line each).
2. `models.streamSimple(model, context)` returned an **event stream**.
3. We iterated it with `for await`, switching on `event.type`, and printed each `text_delta` live so the reply typed itself out.
4. `await stream.result()` gave us the final `AssistantMessage` with `stopReason` and token/cost `usage` — identical structure to `completeSimple`.

Next: **03 — Multi-turn chat** — keep a `Message[]` transcript so the model remembers earlier turns.